# 🍎 APPLE CNN — IMAGE TESTING (INTERNAL 30 vs EXTERNAL 30)
### Project: Plant Disease Detection (Computer Vision)
**Diagnostic Protocol:**
1. **Internal Benchmark (30 images):** 10 Healthy, 10 Apple Scab, 10 Cedar Apple Rust from `.npz` dataset.
2. **External Real-World (30 images):** 10 Healthy, 10 Apple Scab, 10 Cedar Apple Rust from `7th Test_Images/Apple_RealWorld/`.

In [1]:
# ================================================================
# 🍎 APPLE CNN — INTERNAL vs EXTERNAL REAL-WORLD IMAGE TESTING
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

MODEL_PATH = os.path.join(BASE, "6th Trained_Model", "apple_cnn_best.keras")
NPZ_PATH   = os.path.join(BASE, "3rd Preprocessing", "apple_processed_data.npz")
REAL_DIR   = os.path.join(BASE, "7th Test_Images", "Apple_RealWorld")

print("=" * 75)
print("🍎 LOADING MODEL & NPZ DATASET")
print("=" * 75)

model = tf.keras.models.load_model(MODEL_PATH)
data = np.load(NPZ_PATH)
X_all = data['X']
y_all = data['y']
class_names = data['class_names']
source_all = data['source']

# ================================================================
# 1. INTERNAL TEST (30 IMAGES: 10 per class)
# ================================================================
print("\n" + "=" * 75)
print("🏠 1. INTERNAL TEST BENCHMARK (30 IMAGES)")
print("=" * 75)

internal_indices = []
for c_idx in range(len(class_names)):
    indices = np.where(y_all == c_idx)[0]
    np.random.seed(42)
    selected = np.random.choice(indices, size=10, replace=False)
    internal_indices.extend(selected)

X_internal = X_all[internal_indices]
y_internal = y_all[internal_indices]

probs_int = model.predict(X_internal)
preds_int = np.argmax(probs_int, axis=1)
acc_int = np.mean(preds_int == y_internal)

print(f"\n📈 Internal Test Accuracy (30 images): {acc_int*100:.2f}%")
for i, c in enumerate(class_names):
    c_acc = np.mean(preds_int[y_internal == i] == i)
    print(f"  * {c:<18}: {c_acc*100:.2f}% ({np.sum(preds_int[y_internal == i] == i)}/10)")

# ================================================================
# 2. EXTERNAL / REAL-WORLD TEST (30 IMAGES)
# ================================================================
print("\n" + "=" * 75)
print("🌍 2. EXTERNAL REAL-WORLD IMAGE TESTING (30 IMAGES)")
print("=" * 75)

real_records = []
for c_idx, c_name in enumerate(class_names):
    folder = os.path.join(REAL_DIR, c_name)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:10]
        for f in files:
            fpath = os.path.join(folder, f)
            with Image.open(fpath) as img:
                img_prep = img.convert("RGB").resize((224, 224))
                arr = np.expand_dims(np.array(img_prep, dtype=np.uint8), axis=0)
                pred_prob = model.predict(arr, verbose=0)
                pred_class = np.argmax(pred_prob[0])
                conf = np.max(pred_prob[0])

                real_records.append({
                    "Filename": f,
                    "Actual_Class": c_name,
                    "Predicted_Class": class_names[pred_class],
                    "Confidence": round(float(conf), 4),
                    "Correct": (pred_class == c_idx)
                })

if real_records:
    df_real = pd.DataFrame(real_records)
    acc_ext = df_real['Correct'].mean()
    print(f"\n📈 External Real-World Accuracy ({len(df_real)} images): {acc_ext*100:.2f}%")
    print(df_real.to_string(index=False))
else:
    print("ℹ️ Add external test photos to 7th Test_Images/Apple_RealWorld/ (10 Healthy, 10 Apple_Scab, 10 Cedar_Apple_Rust).")


Mounted at /content/drive
🍎 LOADING MODEL & NPZ DATASET

🏠 1. INTERNAL TEST BENCHMARK (30 IMAGES)
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step

📈 Internal Test Accuracy (30 images): 96.67%
  * Healthy           : 90.00% (9/10)
  * Apple_Scab        : 100.00% (10/10)
  * Cedar_Apple_Rust  : 100.00% (10/10)

🌍 2. EXTERNAL REAL-WORLD IMAGE TESTING (30 IMAGES)

📈 External Real-World Accuracy (30 images): 56.67%
                                                                                                  Filename     Actual_Class  Predicted_Class  Confidence  Correct
                                                                          Screenshot 2026-09-05 021015.png          Healthy       Apple_Scab      0.6320    False
                                                                          Screenshot 2026-09-05 021104.png          Healthy          Healthy      0.9999     True
                                                                          Screenshot 2026-09-05 021305.png      